<a href="https://colab.research.google.com/github/athitthiyan/Learning_Gen_AI/blob/main/Colab2_Orchestration_Patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab 2 · Orchestration Patterns in Depth
### Day 19 — Agent Orchestration with AutoGen Studio & Semantic Kernel

Colab 1 used the simplest pattern (RoundRobin). Now you'll wire the **same research team four different ways** and watch how the *control flow* changes — then peek at the **same idea in Semantic Kernel**.

**You will build:**
1. **SelectorGroupChat** — an LLM decides who speaks next.
2. **Swarm** — agents hand off to each other directly.
3. **GraphFlow** — a deterministic researcher → writer → reviewer graph.
4. A **function tool** the researcher calls to delegate real work.
5. A **Semantic Kernel** sequential-orchestration mini-example.

⏱️ ~60 min including the extension tasks at the end.

> The patterns are the lesson. AutoGen, Semantic Kernel and the Microsoft Agent Framework all expose this same family — RoundRobin/Sequential, Selector/GroupChat, Swarm/Handoff, Graph, Magentic.

## 0 · Setup

In [1]:
%pip install -q -U "autogen-agentchat" "autogen-ext[openai]"
print("AutoGen installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 13.1 MB/s eta 0:00:00
AutoGen installed.


In [3]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Ready.")

Ready.


### Three specialists we'll reuse

Notice the **descriptions** — Selector and Swarm route work based on them, so they have to be specific and non-overlapping.

In [4]:
def make_specialists():
    planner = AssistantAgent(
        name="planner",
        model_client=model_client,
        description="Breaks a topic into 2-3 concrete sub-questions to research.",
        system_message="You plan research. Given a topic, list 2-3 specific sub-questions. Keep it short.",
    )
    researcher = AssistantAgent(
        name="researcher",
        model_client=model_client,
        description="Answers factual sub-questions with concise bullet points.",
        system_message="You answer the planner's sub-questions with short factual bullets.",
    )
    writer = AssistantAgent(
        name="writer",
        model_client=model_client,
        description="Turns research bullets into a tight 4-sentence summary, ending with APPROVE.",
        system_message="Write a tight 4-sentence summary from the research. End your message with APPROVE.",
    )
    return planner, researcher, writer

print("Specialist factory ready.")

Specialist factory ready.


## 1 · SelectorGroupChat — let an LLM route

Instead of a fixed order, a **SelectorGroupChat** uses a model to pick *who should act next* based on the conversation and each agent's `description`. Good when the next best speaker depends on what just happened.

Key knobs: it needs its own `model_client` to do the routing, and `allow_repeated_speaker=False` stops one agent from monopolising the floor.

In [5]:
from autogen_agentchat.teams import SelectorGroupChat

planner, researcher, writer = make_specialists()
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

selector_team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,        # the "router" brain
    termination_condition=termination,
    allow_repeated_speaker=False,
)

await Console(selector_team.run_stream(
    task="Topic: why are reusable cups better than disposable ones?"
))

---------- TextMessage (user) ----------
Topic: why are reusable cups better than disposable ones?
---------- TextMessage (planner) ----------
1. What environmental impact do reusable cups have compared to disposable cups in terms of waste reduction?
2. How do the cost savings of using reusable cups accumulate over time compared to purchasing disposable cups?
3. What health benefits are associated with reusable cups as opposed to disposable ones made from single-use plastics?
---------- TextMessage (researcher) ----------
1. **Environmental Impact and Waste Reduction:**
   - Reusable cups significantly reduce the volume of waste sent to landfills, as they can be used multiple times.
   - Production of disposable cups often involves deforestation, water consumption, and greenhouse gas emissions, which reusable cups can help mitigate.
   - Many disposable cups are not recyclable due to their plastic lining, leading to increased environmental harm.

2. **Cost Savings Over Time:**
   - Ini

TaskResult(messages=[TextMessage(id='b23981e0-0644-4347-9c6f-3c4a9425ed04', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 9, 374362, tzinfo=datetime.timezone.utc), content='Topic: why are reusable cups better than disposable ones?', type='TextMessage'), TextMessage(id='d6dff3f6-9709-4359-a4a1-cf2d2a6bfac5', source='planner', models_usage=RequestUsage(prompt_tokens=45, completion_tokens=60), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 13, 755034, tzinfo=datetime.timezone.utc), content='1. What environmental impact do reusable cups have compared to disposable cups in terms of waste reduction?\n2. How do the cost savings of using reusable cups accumulate over time compared to purchasing disposable cups?\n3. What health benefits are associated with reusable cups as opposed to disposable ones made from single-use plastics?', type='TextMessage'), TextMessage(id='9588e076-f578-4186-a46e-a40a336be19e', source='researcher', m

Look at the speaker order in the transcript — it was **chosen at runtime**, not fixed. That's the difference from RoundRobin.

## 2 · Swarm — agents hand off to each other

In a **Swarm**, control is decentralised: each agent declares who it can **hand off** to via `handoffs=[...]`, and passes control with a handoff message. There's no central router — the agents themselves decide.

We'll build a tiny triage flow: a `triage` agent routes to either `billing` or `tech`, and those can hand back to the user when done.

In [6]:
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination

triage = AssistantAgent(
    name="triage",
    model_client=model_client,
    handoffs=["billing", "tech"],
    description="Front desk: routes the user to the right specialist.",
    system_message="Decide if the request is about billing or tech, then hand off to that agent.",
)
billing = AssistantAgent(
    name="billing",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles billing and refund questions.",
    system_message="Answer the billing question. If it's not billing, hand back to triage.",
)
tech = AssistantAgent(
    name="tech",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles technical troubleshooting.",
    system_message="Answer the tech question concisely, then say DONE.",
)

swarm = Swarm(
    participants=[triage, billing, tech],          # Swarm starts with the first agent
    termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(8),
)

await Console(swarm.run_stream(task="My app keeps crashing when I open the camera."))

---------- TextMessage (user) ----------
My app keeps crashing when I open the camera.
---------- ToolCallRequestEvent (triage) ----------
[FunctionCall(id='call_4GxePVqvQtHQuAUwLqKzGiTe', arguments='{}', name='transfer_to_tech')]
---------- ToolCallExecutionEvent (triage) ----------
[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='transfer_to_tech', call_id='call_4GxePVqvQtHQuAUwLqKzGiTe', is_error=False)]
---------- HandoffMessage (triage) ----------
Transferred to tech, adopting the role of tech immediately.
---------- ToolCallRequestEvent (tech) ----------
[FunctionCall(id='call_RMji9iBIsZ5Sbd8imQXo3krg', arguments='{}', name='transfer_to_triage')]
---------- ToolCallExecutionEvent (tech) ----------
[FunctionExecutionResult(content='Transferred to triage, adopting the role of triage immediately.', name='transfer_to_triage', call_id='call_RMji9iBIsZ5Sbd8imQXo3krg', is_error=False)]
---------- HandoffMessage (tech) ----------
Trans

TaskResult(messages=[TextMessage(id='64e616b7-9d6c-455c-a07b-9b02ad2539c5', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 29, 576279, tzinfo=datetime.timezone.utc), content='My app keeps crashing when I open the camera.', type='TextMessage'), ToolCallRequestEvent(id='7e3ecf16-91e2-4314-a324-7e34b16695a4', source='triage', models_usage=RequestUsage(prompt_tokens=82, completion_tokens=12), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 30, 448961, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_4GxePVqvQtHQuAUwLqKzGiTe', arguments='{}', name='transfer_to_tech')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='80d2f44a-4583-48b2-9ec4-e29a47bf1e3c', source='triage', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 30, 450540, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='tra

Watch for the **HandoffMessage** in the transcript — that's one agent explicitly delegating to another. `HandoffTermination(target="user")` is another common stop condition when an agent hands control back to a human.

## 3 · GraphFlow — a deterministic workflow

When you need the **same path every time** (auditable, reproducible), use **GraphFlow**. You declare nodes and directed edges with `DiGraphBuilder`; execution follows the graph exactly.

Here: `planner → researcher → writer`, a fixed pipeline with no LLM routing brain.

In [7]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

planner, researcher, writer = make_specialists()

builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer)
builder.add_edge(planner, researcher).add_edge(researcher, writer)
graph = builder.build()

flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

await Console(flow.run_stream(task="Topic: the benefits of cycling to work."))

---------- TextMessage (user) ----------
Topic: the benefits of cycling to work.
---------- TextMessage (planner) ----------
1. How does cycling to work impact physical health compared to other forms of commuting?  
2. What are the environmental benefits of increased cycling to work in urban areas?  
3. How does cycling to work influence productivity and mental well-being in employees?
---------- TextMessage (researcher) ----------
1. **Physical Health Benefits of Cycling to Work**:  
   - Increases cardiovascular fitness.  
   - Helps with weight management and reduces obesity risk.  
   - Strengthens muscles and improves joint mobility.  
   - Lowers the risk of chronic diseases such as diabetes and heart disease.  
   - Provides a low-impact exercise alternative compared to running/jogging.  

2. **Environmental Benefits of Increased Cycling**:  
   - Reduces greenhouse gas emissions and air pollution.  
   - Decreases traffic congestion on urban roads.  
   - Lowers noise pollution

TaskResult(messages=[TextMessage(id='7ab273df-79c3-4117-b3ac-21a151088304', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 42, 796953, tzinfo=datetime.timezone.utc), content='Topic: the benefits of cycling to work.', type='TextMessage'), TextMessage(id='5ad90af3-730c-4f65-bf87-9f68130a0527', source='planner', models_usage=RequestUsage(prompt_tokens=43, completion_tokens=51), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 43, 837874, tzinfo=datetime.timezone.utc), content='1. How does cycling to work impact physical health compared to other forms of commuting?  \n2. What are the environmental benefits of increased cycling to work in urban areas?  \n3. How does cycling to work influence productivity and mental well-being in employees?', type='TextMessage'), TextMessage(id='800ce2f3-9489-4a52-a90e-2c62345b5b55', source='researcher', models_usage=RequestUsage(prompt_tokens=91, completion_tokens=236), metadata={}, created_at=

GraphFlow gives you **determinism**: the order is guaranteed by the graph, not decided by a model. That's exactly what you want for a compliance-sensitive or repeatable pipeline.

## 4 · A function tool the researcher can call

Delegation isn't only agent-to-agent — an agent can delegate to **code** via a tool. Define a plain Python function, pass it in `tools=[...]`, and the agent will call it when useful. (Here it's a stub; swap in a real search API at home.)

In [8]:
def web_search(query: str) -> str:
    """Look up a query and return a short text snippet. (Stub for the workshop.)"""
    canned = {
        "reusable cup co2": "A reusable cup typically breaks even vs. disposables after ~20-100 uses.",
        "default": "No exact match; returning a generic note that reusable goods amortise their footprint with use.",
    }
    return canned.get(query.lower().strip(), canned["default"])

researcher_with_tool = AssistantAgent(
    name="researcher",
    model_client=model_client,
    tools=[web_search],
    description="Researches facts, calling web_search when it needs evidence.",
    system_message="Use the web_search tool to find a figure, then report it in one bullet. End with APPROVE.",
)

from autogen_agentchat.teams import RoundRobinGroupChat
tool_team = RoundRobinGroupChat(
    [researcher_with_tool],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(4),
)
await Console(tool_team.run_stream(task="Find a figure on reusable cup CO2 break-even and report it."))

---------- TextMessage (user) ----------
Find a figure on reusable cup CO2 break-even and report it.
---------- ToolCallRequestEvent (researcher) ----------
[FunctionCall(id='call_C0kjiPhzFlQQlVzfUmPLFbvG', arguments='{"query":"reusable cup CO2 break-even figure"}', name='web_search')]
---------- ToolCallExecutionEvent (researcher) ----------
[FunctionExecutionResult(content='No exact match; returning a generic note that reusable goods amortise their footprint with use.', name='web_search', call_id='call_C0kjiPhzFlQQlVzfUmPLFbvG', is_error=False)]
---------- ToolCallSummaryMessage (researcher) ----------
No exact match; returning a generic note that reusable goods amortise their footprint with use.
---------- TextMessage (researcher) ----------
Reusable cups typically amortize their carbon footprint over time with repeated use, but specific figures on CO2 break-even points may vary depending on materials and usage frequency. 

APPROVE


TaskResult(messages=[TextMessage(id='5e158a44-6d09-44f7-b55d-af1a173f3c6b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 55, 526321, tzinfo=datetime.timezone.utc), content='Find a figure on reusable cup CO2 break-even and report it.', type='TextMessage'), ToolCallRequestEvent(id='8cc49efa-895b-41c6-9d64-f1d6a36e88ba', source='researcher', models_usage=RequestUsage(prompt_tokens=97, completion_tokens=21), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 56, 304489, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_C0kjiPhzFlQQlVzfUmPLFbvG', arguments='{"query":"reusable cup CO2 break-even figure"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='ef4201cf-7ec7-45ff-b6dc-7985d474bd2c', source='researcher', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 21, 56, 308087, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='No exact ma

The transcript shows a **ToolCall** and its result — the agent delegated part of its job to your function. In production you'd scope tools to least privilege and validate their arguments.

## 5 · The same idea in Semantic Kernel

Semantic Kernel expresses these patterns too — its **Sequential** orchestration is the SK analogue of RoundRobin/GraphFlow-in-a-line. The code below shows the *shape* of SK agent orchestration.

> ⚠️ SK's agent-orchestration API is newer and evolving (and SK is in maintenance mode heading into the Microsoft Agent Framework). If an import path has moved, check the official Semantic Kernel docs — the **concept** is what transfers, not the exact symbol names.

In [9]:
%pip install -q -U semantic-kernel
print("Semantic Kernel installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.3/929.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.9/217.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [10]:
# The shape of an SK sequential orchestration: two agents, output of one feeds the next.
# Wrapped in try/except because SK's orchestration symbols move between versions.
import asyncio

try:
    from semantic_kernel.agents import ChatCompletionAgent
    from semantic_kernel.agents.orchestration.sequential import SequentialOrchestration
    from semantic_kernel.agents.runtime import InProcessRuntime
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service = OpenAIChatCompletion(ai_model_id="gpt-4o-mini")

    sk_writer = ChatCompletionAgent(
        name="writer", service=service,
        instructions="Write one short paragraph on the given topic.",
    )
    sk_editor = ChatCompletionAgent(
        name="editor", service=service,
        instructions="Tighten the paragraph you receive into two crisp sentences.",
    )

    orchestration = SequentialOrchestration(members=[sk_writer, sk_editor])
    runtime = InProcessRuntime()
    runtime.start()

    result = await orchestration.invoke(
        task="The benefits of walking meetings.", runtime=runtime
    )
    print(await result.get())
    await runtime.stop_when_idle()

except Exception as e:
    print("SK orchestration symbols may have moved in your installed version.")
    print("Concept: members=[writer, editor] run in sequence, output -> input.")
    print("Check https://learn.microsoft.com/semantic-kernel for the current API.")
    print("Error was:", type(e).__name__, e)

Walking meetings combine physical activity with collaboration, enhancing health and productivity. They promote creativity, reduce stress, and strengthen team relationships, all while refreshing energy and improving problem-solving.


Notice the **identical mental model**: a list of agents, run in order, each one's output feeding the next. RoundRobin (AutoGen) ≈ Sequential (SK) ≈ a linear GraphFlow. Learn it once.

---
## 🚀 Extension tasks

### Extension 1 — Write a custom selector function
`SelectorGroupChat` accepts a `selector_func` that overrides the LLM router with your own logic. Write a function that **forces** `planner` to go first, then lets the model choose. (Signature: it receives the message history and returns the next speaker's name, or `None` to defer to the model.)

### Extension 2 — Add a conditional GraphFlow edge
Extend the graph from §3 with a **reviewer** node and a **conditional edge**: if the writer's output contains the word `REVISE`, loop back to the writer; otherwise finish. Use `DiGraphBuilder`'s conditional-edge support and a stop condition so it can't loop forever.

### Extension 3 — Nest a team inside a graph node
A node in a GraphFlow can itself be a **team**. Replace the single `researcher` node with a 2-agent `RoundRobinGroupChat` (researcher + fact-checker) and wire that team in as one node. This is *composition*: patterns nest inside patterns.

Scaffolds below.

In [11]:
# === Extension 1 scaffold: custom selector_func ===
def force_planner_first(messages):
    # Return an agent name (str) to force a speaker, or None to let the model decide.
    if len(messages) <= 1:
        return "planner"
    return None

planner, researcher, writer = make_specialists()
team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,
    selector_func=force_planner_first,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
)
await Console(team.run_stream(task="Topic: benefits of a standing desk."))

---------- TextMessage (user) ----------
Topic: benefits of a standing desk.
---------- TextMessage (planner) ----------
1. How does using a standing desk impact productivity and focus during work hours?  
2. What are the long-term health effects of transitioning from a traditional desk to a standing desk?  
3. How does the use of a standing desk affect posture and musculoskeletal health?
---------- TextMessage (researcher) ----------
1. **Impact on Productivity and Focus:**
   - May enhance energy levels and reduce fatigue.
   - Can lead to improved mood and decreased feelings of stress.
   - Some studies indicate increased focus and engagement during tasks.

2. **Long-term Health Effects:**
   - Potential reduction in risks of obesity and related diseases.
   - May lower the likelihood of developing cardiovascular issues.
   - Long-term use can help in managing blood sugar levels.

3. **Effect on Posture and Musculoskeletal Health:**
   - Encourages a more neutral spinal alignment.
 

TaskResult(messages=[TextMessage(id='287d2587-3dd6-4d14-ae21-e3707596382a', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 22, 57, 123963, tzinfo=datetime.timezone.utc), content='Topic: benefits of a standing desk.', type='TextMessage'), TextMessage(id='e08e00ab-2202-495d-b75d-15d300b93a5f', source='planner', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=55), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 22, 58, 182617, tzinfo=datetime.timezone.utc), content='1. How does using a standing desk impact productivity and focus during work hours?  \n2. What are the long-term health effects of transitioning from a traditional desk to a standing desk?  \n3. How does the use of a standing desk affect posture and musculoskeletal health?', type='TextMessage'), TextMessage(id='2d739e4d-42ca-4a05-a212-33678e06edab', source='researcher', models_usage=RequestUsage(prompt_tokens=94, completion_tokens=140), metadata={}, created_

In [15]:
builder = DiGraphBuilder()
reviewer = AssistantAgent(
    name="reviewer",
    model_client=model_client,
    system_message="You are a strict reviewer. Check the writer's draft. If it needs changes, include the word REVISE in your response along with specific feedback. If it's good, approve it without using the word REVISE."
)

# Define a 'finalizer' agent to act as a leaf node for successful termination
finalizer = AssistantAgent(
    name="finalizer",
    model_client=model_client,
    system_message="This agent marks the successful completion of the review process. Your only task is to say 'APPROVED' and nothing else."
)

builder.add_node(planner).add_node(researcher).add_node(writer).add_node(reviewer).add_node(finalizer)
builder.add_edge(planner, researcher).add_edge(researcher, writer).add_edge(writer, reviewer)

# Conditional: reviewer -> writer only if "REVISE" appears
builder.add_edge(reviewer, writer, condition=lambda msg: "REVISE" in msg.to_text())
# Conditional: reviewer -> finalizer if "REVISE" does NOT appear (the "otherwise finish" part)
builder.add_edge(reviewer, finalizer, condition=lambda msg: "REVISE" not in msg.to_text())

graph = builder.build()
flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
    # Added MaxMessageTermination to prevent infinite loops in case of continuous revisions
    termination_condition=TextMentionTermination("APPROVED") | MaxMessageTermination(10)
)

# Example run for the updated flow
await Console(flow.run_stream(task="Topic: the importance of clear communication in teams."))

---------- TextMessage (user) ----------
Topic: the importance of clear communication in teams.
---------- TextMessage (planner) ----------
1. How does clear communication influence team collaboration and decision-making?  
2. What are the effects of miscommunication on team dynamics and project outcomes?  
3. What strategies can teams implement to improve communication effectiveness?  
---------- TextMessage (researcher) ----------
1. **Influence on Team Collaboration and Decision-Making:**
   - Facilitates alignment on goals and objectives.
   - Enhances trust and openness among team members.
   - Leads to faster and more informed decision-making.

2. **Effects of Miscommunication:**
   - Can create confusion and misunderstandings among team members.
   - Leads to delays and inefficiencies in project timelines.
   - May contribute to conflict and reduced morale within the team.

3. **Strategies to Improve Communication Effectiveness:**
   - Establish regular check-ins and updates amo

TaskResult(messages=[TextMessage(id='674f45b3-2287-4360-8c56-80583129377d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 27, 20, 226813, tzinfo=datetime.timezone.utc), content='Topic: the importance of clear communication in teams.', type='TextMessage'), TextMessage(id='05ecb8f6-992c-4c7e-95bb-fa86ce285d02', source='planner', models_usage=RequestUsage(prompt_tokens=116, completion_tokens=44), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 27, 21, 118014, tzinfo=datetime.timezone.utc), content='1. How does clear communication influence team collaboration and decision-making?  \n2. What are the effects of miscommunication on team dynamics and project outcomes?  \n3. What strategies can teams implement to improve communication effectiveness?  ', type='TextMessage'), TextMessage(id='963740cd-9163-4aec-aee7-6406841cbfa0', source='researcher', models_usage=RequestUsage(prompt_tokens=303, completion_tokens=151), metadata={}, created_

In [17]:
# === Extension 3 scaffold: nest a team inside a node ===
fact_checker = AssistantAgent(
    name="fact_checker",
    model_client=model_client,
    system_message="You are a meticulous fact-checker. Verify the researcher's claims for accuracy. Flag any unsupported or questionable statements clearly."
)
research_team = RoundRobinGroupChat(
    [researcher, fact_checker],
    termination_condition=MaxMessageTermination(4),
)
builder = DiGraphBuilder()
builder.add_node(planner).add_node(research_team).add_node(writer)
builder.add_edge(planner, research_team).add_edge(research_team, writer)

## Recap

You orchestrated one research team **five ways** and saw exactly how control flow differs:

* **Selector** — LLM picks the next speaker (dynamic).
* **Swarm** — agents hand off to each other (decentralised).
* **GraphFlow** — a fixed, auditable graph (deterministic).
* **Tools** — an agent delegates work to a function.
* **Semantic Kernel** — the same Sequential idea in the enterprise SDK.

Pair this with the decision matrix from the slides, then tackle the **capstone**: build your own delegating team in AutoGen Studio.

In [18]:
await model_client.close()
print("Client closed. On to the capstone!")

Client closed. On to the capstone!
